# Night-time lights per capita

Nine years of VIIRS night-time lights per person for 339 municipalities, with a long-run trend: a satellite proxy for local economic activity.

This notebook gathers every code box of the indicator's page, [https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities339-ntl/](https://quarcs-lab.github.io/gdo4bolivia/indicators/municipalities339-ntl/) (GeoDevelopment Observatory of Bolivia), so you can work with the data in one place. Run **Setup** first; then any section runs on its own (Runtime › Run all runs everything).

## Ask Gemini, Colab's AI

Colab has an AI assistant, Gemini. Open it with the Gemini button (✦) in Colab's toolbar, or click **Generate** in an empty code cell, and paste one of these questions. It writes and explains code that builds on the data loaded below. Run Setup first, and check its answers against the variables' descriptions on the page.

- "Plot log night-time lights per capita (ln_NTLpc) by department from 2012 to 2020 with geo.trend("ntl", "ln_NTLpc", by="department") and describe the trends."
- "Map the change of the trend (ln_t400NTLpc) between 2012 and 2020 and list the 10 municipalities that grew most."
- "Did night-time lights per capita form convergence clubs? Run geo.convergence_clubs("ntl", "ln_NTLpc"), map the clubs and explain them."
- "Are lights spatially clustered? Run geo.moran and geo.lisa on ln_NTLpc in 2020 and describe the clusters on a map."
- "Merge ntl with pop (both on map339) and plot ln_NTLpc against population density in 2020 (geo.combine)."


## Variables

The columns of `ntl`, one row per unit and year. Each one's full description (how it was built, its source and reference years) is under Variables on the page, and in `geo.dictionary("ntl")`.

| Variable | Label | Unit |
| --- | --- | --- |
| `geo_id` | Municipality ID |  |
| `year` | Year |  |
| `asdf_id` | DS4Bolivia polygon ID |  |
| `ln_NTLpc` | Log night-time lights per capita | log(10⁸ × nW/cm²/sr per person) |
| `ln_t400NTLpc` | Log night-time lights per capita (trend) | log(10⁸ × nW/cm²/sr per person) |


## Setup

Installs the packages every section uses and imports them once.

In [ ]:
!pip install -q geo4bolivia geolibre
import geo4bolivia as geo
import plotly.express as px
import pandas as pd
import geopandas as gpd
from geolibre import Map

## Load the data with the package

In [ ]:
df = geo.load("ntl")
df = df.merge(geo.units("map339")[["geo_id", "department_id"]], on="geo_id")

# Long-run trend of lights per capita: mean by department
trend = df.groupby(["year", "department_id"], as_index=False)["ln_t400NTLpc"].mean()
names = geo.units("department").set_index("geo_id")["name"]
trend["department"] = trend["department_id"].map(names)
px.line(trend, x="year", y="ln_t400NTLpc", color="department",
labels={"ln_t400NTLpc": "Log lights per capita (trend)"},
title="Night-time lights by department")

## Load the data without the package (plain URLs)

In [ ]:
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
df = pd.read_csv(BASE + "data/ntl/map339/ntl.csv")
units = pd.read_csv(BASE + "geo/units.csv")   # every scale and map
units = units[units["scale"].isin(["map339", "department"])]
df = df.merge(units[["geo_id", "department_id"]], on="geo_id")

# Long-run trend of lights per capita: mean by department
trend = df.groupby(["year", "department_id"], as_index=False)["ln_t400NTLpc"].mean()
trend["department"] = trend["department_id"].map(units.set_index("geo_id")["name"])
px.line(trend, x="year", y="ln_t400NTLpc", color="department",
labels={"ln_t400NTLpc": "Log lights per capita (trend)"},
title="Night-time lights by department")

## Load the data from the wide GeoPackage (one row per municipality)

In [ ]:
# No geo4bolivia needed: one row per municipality, a column per variable
# and year (<variable>_<year>), with boundaries
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
gdf = gpd.read_file(BASE + "data/ntl/map339/ntl_wide.gpkg",
                    layer="ntl")   # or ntl_wide.geojson (lighter)
px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index,
          color="ln_NTLpc_2020", hover_name="name",
          center={"lat": -16.3, "lon": -64}, zoom=4,
          labels={"ln_NTLpc_2020": "Log night-time lights per capita"},
          title="Log night-time lights per capita, 2020")

## Load the data in GeoLibre (the wide GeoJSON)

In [ ]:
# GeoLibre in the notebook: the wide GeoJSON, one feature per municipality
# and a column per variable and year (<variable>_<year>); restyle any of
# them in the Style panel
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"
m = Map(center=[-64.5, -16.5], zoom=4.5, height="600px")
m.add_choropleth(BASE + "data/ntl/map339/ntl_wide.geojson",
                 column="ln_NTLpc_2020", name="Log night-time lights per capita, 2020",
                 strokeColor="#ffffff", strokeWidth=0.3)
m

## License and how to cite

The data are published under [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/); their sources keep their own licenses:

- Earth Observation Group, Payne Institute for Public Policy, Colorado School of Mines: [Creative Commons Attribution 4.0 International License](https://eogdata.mines.edu/files/EOG_products_CC_License.pdf)
- WorldPop (University of Southampton, University of Louisville, Université de Namur) and CIESIN, Columbia University: [Creative Commons Attribution 4.0 International License](https://hub.worldpop.org/doi/10.5258/SOTON/WP00647)
- AidData, William & Mary: [Open Data Commons Attribution License 1.0](https://opendatacommons.org/licenses/by/1-0/)
- geoBoundaries, William & Mary geoLab: [Creative Commons Attribution 4.0 International License](https://creativecommons.org/licenses/by/4.0/)
- Carlos Mendez, Erick Gonzales, Pedro Leoni, Lykke Andersen and Hernán Peralta: [MIT License](https://github.com/quarcs-lab/ds4bolivia/blob/master/LICENSE)

If you use these data, please cite, in this order, the original source, the observatory, which processes and maintains them, and the other sources:

1. Elvidge, C. D., Zhizhin, M., Ghosh, T., Hsu, F.-C., & Taneja, J. (2021). Annual time series of global VIIRS nighttime lights derived from monthly averages: 2012 to 2019. Remote Sensing, 13(5), 922.
2. WorldPop & CIESIN (2018). Global High Resolution Population Denominators Project (1 km aggregated, 2000–2020). University of Southampton and Columbia University. Funded by the Bill and Melinda Gates Foundation (OPP1134076).
3. Mendez, C. (2026). GeoDevelopment Observatory of Bolivia. https://quarcs-lab.github.io/gdo4bolivia/
4. Goodman, S., BenYishay, A., Lv, Z., & Runfola, D. (2019). GeoQuery: Integrating HPC systems and public web-based geospatial data tools. Computers & Geosciences, 122, 103–112.
5. Runfola, D., Anderson, A., Baier, H., et al. (2020). geoBoundaries: A global database of political administrative boundaries. PLoS ONE, 15(4), e0231866.
6. Mendez, C., Gonzales, E., Leoni, P., Andersen, L., & Peralta, H. (2026). DS4Bolivia: A Data Science Repository to Study GeoSpatial Development in Bolivia [Data set]. GitHub.
